# DAIC-WOZ full-188: seed 42, from start to finish
Select a Colab GPU runtime. Put your existing GitHub token in Colab Secrets as `GITHUB_TOKEN`. All new persistent results go under `/content/drive/MyDrive/DAIC_WOZ/experiments/full188_seed42/`. Keep the final student test cell closed until one mode is chosen on validation.


In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import base64, subprocess, sys

drive.mount('/content/drive')
DAIC = Path('/content/drive/MyDrive/DAIC_WOZ')
assert DAIC.is_dir(), f'Missing {DAIC}'
REPO = 'engrkhubabahmad/reliability-aware-depression-kd'
BRANCH = 'feature/full188-fresh-teachers-student-seed42'
ROOT = Path('/content/reliability-aware-depression-kd')
CODE = Path('/content/reliability-aware-depression-kd-full188')
token = userdata.get('GITHUB_TOKEN')
assert token, 'Add GITHUB_TOKEN to Colab Secrets'
auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
git = ['git', '-c', f'http.extraHeader=AUTHORIZATION: basic {auth}']
if not (ROOT / '.git').exists():
    subprocess.run(git + ['clone', f'https://github.com/{REPO}.git', str(ROOT)], check=True)
subprocess.run(git + ['-C', str(ROOT), 'fetch', 'origin', BRANCH], check=True)
revision = subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'FETCH_HEAD'], text=True).strip()
if not CODE.exists():
    subprocess.run(['git', '-C', str(ROOT), 'worktree', 'add', '--detach', str(CODE), revision], check=True)
else:
    assert (CODE / '.git').exists() and not subprocess.check_output(['git', '-C', str(CODE), 'status', '--porcelain'], text=True).strip()
    subprocess.run(['git', '-C', str(CODE), 'switch', '--detach', revision], check=True)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(CODE / 'requirements-colab.txt')])
del token, auth, git
print('Code revision:', revision)


## Run all stages
The runner checks inputs, creates the split and features, and trains the teachers and students. It runs directly in the notebook so epoch bars render in Colab. Completed stages are skipped. With `--archive-incomplete`, any earlier incomplete stage is moved under `failed_attempts/` before retrying; its files remain available for diagnosis.


In [ ]:
from scripts.experiments.full188.run_all import main as run_full188
run_full188(['--daic-root', str(DAIC), '--archive-incomplete'])


## Review validation and teacher outputs
Use validation to choose one mode. In-sample TRAIN metrics are diagnostic only.


In [ ]:
import pandas as pd
EXP = DAIC / 'experiments/full188_seed42'
for path in [EXP / 'teachers/text/train_text_targets.csv',
             EXP / 'teachers/audio/train_audio_targets.csv',
             EXP / 'students/fusion/val_missing_noise.csv']:
    print(path, 'exists:', path.is_file())
    if path.is_file(): display(pd.read_csv(path).head())
print((EXP / 'students/fusion/audit.json').read_text())


## Final student test: run once after freezing a mode
Change `CHOSEN_MODE` to the selected validation mode, then run this cell only once. The original AVEC test partition has been redistributed, so this is an internal held-out test.


In [ ]:
CHOSEN_MODE = 'ra_kd'  # Set once based on validation; options: no_kd, standard_kd, ra_kd
subprocess.run([sys.executable, '-m', 'scripts.experiments.full188.score_student_test',
                '--daic-root', str(DAIC), '--split-dir', str(EXP / 'split'),
                '--features', str(EXP / 'features'),
                '--student-branches', str(EXP / 'students/branches'),
                '--audio-teacher', str(EXP / 'teachers/audio'),
                '--student-fusion', str(EXP / 'students/fusion'),
                '--mode', CHOSEN_MODE, '--output', str(EXP / 'students/internal_test')],
               cwd=CODE, check=True)
print((EXP / 'students/internal_test/result.json').read_text())
